# PlantCLEF 2015 Leaf S-CNN Training

Clean Colab workflow for the current project state. Use the Google Drive leaf-only archive first, then smoke-train, evaluate `S-CNN (A)`, and only then run longer training.

## 1. Runtime Check

Select `Runtime -> Change runtime type -> GPU` before running training cells.

In [ ]:
import torch

print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 2. Clone Or Update Project

In [ ]:
%cd /content
!rm -rf diploma
!git clone -b robodanill/main https://github.com/robodanill/diploma.git
%cd /content/diploma
!pip install -e '.[ml]'


## 3. Mount Google Drive And Unpack Leaf Dataset

Expected archive path: `/content/drive/MyDrive/PlantCLEF2015_leaf_only.tar.gz`.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
%cd /content/diploma
!rm -rf data/plantclef2015
!mkdir -p data/plantclef2015
!tar -xzf /content/drive/MyDrive/PlantCLEF2015_leaf_only.tar.gz -C data/plantclef2015
!cp data/plantclef2015/leaf/metadata.csv data/plantclef2015/metadata.csv
!wc -l data/plantclef2015/metadata.csv
!find data/plantclef2015/leaf/train -maxdepth 1 -type f -name '*.jpg' | wc -l


## 4. Smoke Train `S-CNN (A)` Genus

This is only a pipeline check on a small subset. It should be fast.

In [ ]:
!plant-classifier-train   --config configs/smoke_training.yaml   --stage genus   --output checkpoints/smoke_scnn_genus_vgg16.pt


## 5. Smoke Evaluate `S-CNN (A)`

In [ ]:
!plant-classifier-eval-genus   --config configs/smoke_training.yaml   --checkpoint checkpoints/smoke_scnn_genus_vgg16.pt   --max-species 40   --references-per-genus 2   --queries-per-genus 2   --top-k 5


## 6. Full Leaf Train `S-CNN (A)` Genus

Use this after the smoke path works. Training now re-samples pairs every epoch and also saves `scnn_genus_vgg16_best.pt`.

In [ ]:
!plant-classifier-train   --config configs/leaf_training.yaml   --stage genus   --output checkpoints/scnn_genus_vgg16.pt


## 7. Evaluate Full `S-CNN (A)`

In [ ]:
!plant-classifier-eval-genus   --config configs/leaf_training.yaml   --checkpoint checkpoints/scnn_genus_vgg16_best.pt   --max-species 120   --references-per-genus 2   --queries-per-genus 2   --top-k 5


## 8. Sync Checkpoints To Google Drive

Old Drive checkpoints are removed unless their name contains `_best`.

In [ ]:
!python scripts/sync_checkpoints_to_drive.py   --source checkpoints   --dest /content/drive/MyDrive/diploma_checkpoints   --keep-token _best
!ls -lh /content/drive/MyDrive/diploma_checkpoints


## 9. Train `S-CNN (B)` Species

Run this only after `S-CNN (A)` has a reasonable genus retrieval result.

In [ ]:
!plant-classifier-train   --config configs/leaf_training.yaml   --stage species   --output checkpoints/scnn_species_vgg16.pt


## 10. Build Reference Index For Desktop Inference

In [ ]:
!plant-classifier-build-index   --config configs/leaf_training.yaml   --genus-checkpoint checkpoints/scnn_genus_vgg16_best.pt   --species-checkpoint checkpoints/scnn_species_vgg16_best.pt   --output checkpoints/reference_index.pt
!ls -lh checkpoints


## 11. Final Sync To Google Drive

In [ ]:
!python scripts/sync_checkpoints_to_drive.py   --source checkpoints   --dest /content/drive/MyDrive/diploma_checkpoints   --keep-token _best
!ls -lh /content/drive/MyDrive/diploma_checkpoints
